# 06 — Limpieza del corpus español (anti-bot, cap por usuario)

Aplica al corpus español los mismos filtros adicionales que se aplicaron al
corpus de EEUU (notebook `02_etl_cleaning`), para que ambos corpus estén
construidos en igualdad de condiciones y la comparación sea metodológicamente
válida.

## Filtros aplicados

1. **Anti-bot por username** — descarta cuentas con patrones de bot/news
   (terminadas en `bot`, `news`, `hub`, `alertas`, etc.).
2. **Cap por usuario** — máximo 10 tweets por usuario, para evitar que una
   sola cuenta (típicamente medios o cuentas institucionales) domine el corpus.
3. **Anti no-España explícito** — descarta ubicaciones claramente de otros
   países hispanohablantes (México, Argentina, Colombia...) que se hayan
   colado pese al filtro previo.

## Entrada / Salida

- Entrada: `scam_es_FINAL_clean.csv` (1.381 tweets)
- Salida: `scam_es_FINAL_v2.csv`

Sin API, sin modelos. Ejecución de segundos.


In [ ]:
import pandas as pd
import re

pd.set_option('display.max_colwidth', None)

df = pd.read_csv("../data/raw/scam_es_FINAL_clean.csv")
print(f"Tweets cargados: {len(df)}")
print(f"\nDistribución inicial por categoría:")
for cat in ['phishing', 'payment_apps', 'crypto', 'romance', 'impersonation']:
    n = df['query_labels'].fillna('').str.contains(cat).sum()
    print(f"  {cat:<16} {n:>5}")


## Filtro 1 — Anti-bot por username

In [ ]:
BOT_NEWS_USERNAME = re.compile(
    r"("
    r"_bot$|bot\d*$|^bot_|"
    r"news$|^news_|noticias?$|"
    r"alerts?$|alertas?$|^alerta_|"
    r"_hub$|hubs?$|"
    r"diario|prensa|"
    r"^auto_|spam$"
    r")",
    re.IGNORECASE,
)

def is_suspected_bot_username(username):
    if not isinstance(username, str) or not username.strip():
        return False
    return bool(BOT_NEWS_USERNAME.search(username))

df["is_bot_username"] = df["username"].apply(is_suspected_bot_username)
print(f"Cuentas marcadas como bot/news: {df['is_bot_username'].sum()}")
print()
print("Ejemplos detectados:")
print(df[df["is_bot_username"]]["username"].value_counts().head(10))


## Filtro 2 — Anti no-España explícito

Descarta ubicaciones de otros países hispanohablantes que se hayan colado.


In [ ]:
NON_SPAIN_LOCATIONS = re.compile(
    r"\b("
    r"mexico|m\u00e9xico|cdmx|guadalajara|monterrey|"
    r"argentina|buenos aires|cordoba\s+argentina|"
    r"colombia|bogota|bogot\u00e1|medellin|medell\u00edn|"
    r"chile|santiago de chile|"
    r"peru|per\u00fa|lima|"
    r"venezuela|caracas|"
    r"ecuador|quito|guayaquil|"
    r"uruguay|montevideo|"
    r"paraguay|asuncion|"
    r"bolivia|la paz|"
    r"guatemala|honduras|salvador|nicaragua|costa rica|panama|panam\u00e1|"
    r"republica dominicana|rep\u00fablica dominicana|santo domingo|"
    r"puerto rico|"
    r"miami|estados unidos|"
    r"usa"
    r")\b",
    re.IGNORECASE,
)

# Indicadores fuertes de España (whitelist por si una ubicación es ambigua)
SPAIN_INDICATORS = re.compile(
    r"\b(espa\u00f1a|spain|madrid|barcelona|valencia|sevilla|bilbao|"
    r"zaragoza|malaga|m\u00e1laga|murcia|galicia|catalu\u00f1a|andalucia|"
    r"andaluc\u00eda|euskadi|asturias)\b",
    re.IGNORECASE,
)

def is_clearly_non_spain(location):
    if not isinstance(location, str) or not location.strip():
        return False
    loc = location.lower()
    if NON_SPAIN_LOCATIONS.search(loc):
        # Si también menciona España explícitamente, no descartar
        if SPAIN_INDICATORS.search(loc):
            return False
        return True
    return False

df["is_non_spain"] = df["user_location"].apply(is_clearly_non_spain)
print(f"Ubicaciones marcadas como no-España: {df['is_non_spain'].sum()}")
if df["is_non_spain"].sum() > 0:
    print()
    print("Ejemplos:")
    print(df[df["is_non_spain"]]["user_location"].value_counts().head(10))


## Aplicar filtros 1 y 2

In [ ]:
df_step1 = df[~(df["is_bot_username"] | df["is_non_spain"])].copy()
print(f"Tras filtros anti-bot y anti-no-España: {len(df_step1)} (era {len(df)})")


## Filtro 3 — Cap por usuario (máximo 10 tweets)

In [ ]:
MAX_TWEETS_PER_USER = 10

# Ver usuarios dominantes antes del cap
print("Top 10 usuarios ANTES del cap:")
print(df_step1["username"].value_counts().head(10))
print()

df_step1 = df_step1.sort_values("created_at", ascending=False)
df_final = df_step1.groupby("username", group_keys=False).head(MAX_TWEETS_PER_USER).reset_index(drop=True)

print(f"Tras cap de {MAX_TWEETS_PER_USER} tweets/usuario: {len(df_final)} (era {len(df_step1)})")
print()
print("Top 10 usuarios DESPUÉS del cap:")
print(df_final["username"].value_counts().head(10))


## Resumen final

In [ ]:
print("=" * 50)
print("   RESUMEN LIMPIEZA CORPUS ESPAÑOL")
print("=" * 50)
print(f"  Partida:                {len(df)}")
print(f"  Tras anti-bot/no-España: {len(df_step1)}")
print(f"  Tras cap por usuario:    {len(df_final)}")
print(f"  Retención:               {len(df_final)/len(df)*100:.1f}%")
print("=" * 50)
print()
print("Distribución final por categoría:")
total = len(df_final)
for cat in ['phishing', 'payment_apps', 'crypto', 'romance', 'impersonation']:
    n = df_final['query_labels'].fillna('').str.contains(cat).sum()
    pct = n / total * 100 if total > 0 else 0
    print(f"  {cat:<16} {n:>5}  ({pct:.1f}%)")


In [ ]:
# Top ubicaciones tras limpieza
print("=== TOP 15 UBICACIONES TRAS LIMPIEZA ===")
print(df_final["user_location"].value_counts().head(15))


## Guardado

In [ ]:
# Limpiar columnas auxiliares
for col in ["is_bot_username", "is_non_spain"]:
    if col in df_final.columns:
        df_final.drop(columns=[col], inplace=True)

OUTPUT = "../data/raw/scam_es_FINAL_v2.csv"
df_final.to_csv(OUTPUT, index=False, encoding="utf-8")
print(f"✓ Guardado: {OUTPUT}")
print(f"  {len(df_final)} tweets, listo para clasificación con mDeBERTa.")
print()
print("🚨 HAZ COPIA DE SEGURIDAD en Drive.")


## Inspección visual

In [ ]:
n = min(20, len(df_final))
print(f"=== MUESTRA DE {n} TWEETS LIMPIOS ===\n")
for _, row in df_final.sample(n, random_state=42).iterrows():
    print(f"[{row['query_labels']}] @{row['username']} — {row['user_location']}")
    print(f"  {str(row['text'])[:280]}")
    print()
